In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(path)

In [ ]:
# Task 2: Write your code here: Inspect the first few rows using head()
df.head()

In [ ]:
# Task 3: Write your code here: Display dataset information using info()
df.info()

In [ ]:
# Task 4: Write your code here: Show statistical description using describe()
df.describe()

In [ ]:
# Task 1: Write your code here: Handle missing values appropriately


df.isnull().sum()

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].mean())

missing_values = df.isnull().sum()
print("Missing Values per Column:")
print(missing_values[missing_values > 0])
if missing_values.any():
  print("\nHandle Missing Values as needed.")
else:
  print("\nNo Missing Values Found.")


In [ ]:
# Task 2: Write your code here: Check and remove duplicates if any exist

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here: Encode categorical variables if needed

categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
df

In [ ]:
# Task 4: Write your code here: Apply feature scaling to numerical features (Use StandardScaler)

from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here: Check for target imbalance and state if it is imbalanced or not

# no need

In [ ]:
# Task 1: Write your code here:

X = df.drop(columns=['Target'])
y = df['Target']

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:



import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier

target_col = "Target"
X = df.drop(columns=[target_col])
y = df[target_col]

counts = y.value_counts()
is_binary = (len(counts) == 2)

if is_binary:
    ratio = counts.max() / counts.min()
    minority_pct = counts.min() / counts.sum()
    imbalanced = (ratio > 1.5) or (minority_pct < 0.40)
else:
    imbalanced = (counts.min() < 0.5 * counts.mean())

n_splits = 5
random_state = 42

if imbalanced:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    metric_name = "F1"
else:
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    metric_name = "Accuracy"

scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.1,
        depth=6,
        loss_function="Logloss",
        verbose=0,
        random_seed=random_state
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    if metric_name == "Accuracy":
        score = accuracy_score(y_val, y_pred)
    else:
        avg = "binary" if is_binary else "macro"
        score = f1_score(y_val, y_pred, average=avg)

    scores.append(score)
    print(f"Fold {fold} {metric_name}: {score:.4f}")

print(f"\nAveraged {metric_name} across {n_splits} folds: {np.mean(scores):.4f}")


In [ ]:
# Task 1: Write your code here:

import matplotlib.pyplot as plt
import pandas as pd

importances = model.get_feature_importance()
feature_names = X.columns

feat_imp = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)


plt.figure(figsize=(10, 6))
plt.barh(feat_imp["Feature"], feat_imp["Importance"])
plt.gca().invert_yaxis()
plt.xlabel("Importance Score")
plt.title("Feature Importance (CatBoost)")
plt.show()


In [ ]:
# Task 2: Write your code here:

import numpy as np

importances = model.get_feature_importance()
feature_names = X.columns

golden_idx = np.argmax(importances)
golden_feature = feature_names[golden_idx]
golden_importance = importances[golden_idx]

print(f"Golden feature: {golden_feature}")
print(f"Importance score: {golden_importance:.4f}")


In [ ]:
# Task Bonus: Write your code here: